# Connection Pool

**Company:** MongoDB (GothamLoop question bank) · **Category:** Coding · **Tags:** Live Screen, Onsite Loop, Concurrency, OOP & Design Patterns · **Difficulty/Frequency:** Very Common (7/10)

> **Language note.** The official answer is written in Java (`Semaphore` + `synchronized`). This notebook implements the *same design* in Python — `threading.Semaphore` for checkout accounting and `threading.Lock` for the idle stack — so every claim below is exercised by real threads. The Java reference is preserved verbatim in [`README.md`](README.md).

## Concepts

**What this problem is really testing:**
- **Lock granularity** — not "is it thread-safe?" but "how *little* can you hold the lock for?"
- The difference between a **mutex** (mutual exclusion, one holder) and a **semaphore** (a counter of permits, n holders)
- The **object pool** pattern: reuse expensive-to-create objects instead of recreating them

**First-principles primer — what is each piece?**

- **Lock / mutex** (`threading.Lock`) — a token exactly one thread can hold. Everyone else waits. It protects a **critical section**: a stretch of code where shared data is briefly inconsistent and must not be observed.
- **Semaphore** (`threading.Semaphore(n)`) — a lock generalised to `n` permits. `acquire()` takes one (blocking if none are free); `release()` gives one back. Where a lock says *"only one at a time"*, a semaphore says *"at most n at a time"* — exactly the shape of "at most `max_size` connections checked out".
- **Critical section** — the code between acquiring and releasing a lock. Everything inside it is serialised, so **its length directly bounds your throughput**. This problem is entirely about making that section as short as possible.
- **Object pool** — when creating an object costs far more than using it (TCP handshake, TLS negotiation, database authentication — often 10–100 ms), you keep a set of them alive and hand them out. Reuse converts a 50 ms cost into a ~0 ms one.

**The one insight the interviewer is listening for:**

> **Never hold a lock while doing slow work.**

The naive answer marks the whole `get_connection` method `synchronized`. It *is* correct. But `open()` takes ~50 ms, so while one thread opens a connection, every other thread is frozen — *even threads that only wanted to grab an idle connection sitting right there in the stack*. The lock has serialised the expensive part of the program.

The fix separates two different jobs that are usually conflated:

| Job | Right tool | Why |
|---|---|---|
| "Are there fewer than `max_size` checked out?" | **Semaphore** | It is a counting question, and it must *block* when the answer is no |
| "Pop/push the idle stack safely" | **Lock**, held for microseconds | Two threads mutating a list must not interleave |
| "Open the connection" | **No lock at all** | Slow, and it touches nothing shared |

**Simple worked example.** Pool of size 2, three threads arrive at once:

| | Thread A | Thread B | Thread C |
|---|---|---|---|
| acquire permit | ✅ (2→1) | ✅ (1→0) | ⛔ **blocks** |
| lock, check stack | empty | empty | — |
| unlock, `open()` (50 ms) | in progress | in progress | — |

A and B open **in parallel** — 50 ms total, not 100 ms. C waits, correctly, because the pool is genuinely full. Then A releases: its permit goes back (0→1), C wakes, finds A's connection idle on the stack, and pays **no** open cost.

## Problem Statement

Implement a thread-safe `ConnectionPool`:

| Method | Behaviour |
|---|---|
| `get_connection()` | Return an open connection: reuse an idle one, or create+open a new one. Block if `max_size` are already checked out |
| `release_connection(conn)` | Return a connection to the pool for reuse |
| `close_all()` | Close every idle connection and reset the pool |

`Connection.open()` is **expensive** and must be called exactly once before any `read`/`write`. After `close()`, the connection is unusable.

**The actual question:** *where do you put the lock for better efficiency?*

In [ ]:
import threading
import time
from typing import List, Optional

OPEN_COST = 0.02          # 20 ms - stands in for a TCP + TLS + auth handshake


class Connection:
    """Mirrors the Java class in the prompt. open() is deliberately slow."""

    _counter = 0
    _counter_lock = threading.Lock()

    def __init__(self) -> None:
        with Connection._counter_lock:
            Connection._counter += 1
            self.id = Connection._counter
        self.opened = False
        self.closed = False

    def open(self) -> None:
        if self.opened:
            raise RuntimeError("open() must be called exactly once")
        time.sleep(OPEN_COST)                  # THE expensive operation
        self.opened = True

    def read(self) -> str:
        self._check()
        return f"data-from-{self.id}"

    def write(self, data: str) -> None:
        self._check()

    def close(self) -> None:
        self.closed = True

    def _check(self) -> None:
        if not self.opened or self.closed:
            raise RuntimeError("connection not usable")


def reset_connection_counter() -> None:
    """So each experiment below can count connections from zero."""
    with Connection._counter_lock:
        Connection._counter = 0

### Approach 1 — Naive (one big lock around the whole method)

**Idea:** mark the entire `get_connection` critical. It is genuinely correct and genuinely thread-safe — no race can occur, because nothing runs concurrently at all.

That is also the problem. `open()` sits **inside** the critical section, so the pool serialises the single slowest operation in the system. With 8 threads that each need a fresh connection, you pay `8 × 20 ms` in sequence instead of `20 ms` in parallel. Worse, a thread that only wanted to *pop an idle connection* — a microsecond of work — waits behind a 20 ms open it has nothing to do with.

It also has no bound: nothing stops it from creating more than `max_size` connections.

**Time complexity:** O(1) of actual work per call, but **effectively serialised** — wall-clock throughput collapses to one open at a time.

**Space complexity:** O(n) for the idle stack.

In [ ]:
class NaiveConnectionPool:
    """Baseline: correct, but the lock is held across the expensive open()."""

    def __init__(self, max_size: int) -> None:
        self.max_size = max_size
        self.available: List[Connection] = []
        self.lock = threading.Lock()
        self.created = 0

    def get_connection(self) -> Connection:
        with self.lock:                       # <-- held for the WHOLE method
            if self.available:
                return self.available.pop()
            conn = Connection()
            conn.open()                       # 20 ms INSIDE the critical section
            self.created += 1
            return conn                       # note: nothing enforces max_size

    def release_connection(self, conn: Connection) -> None:
        with self.lock:
            self.available.append(conn)

    def close_all(self) -> None:
        with self.lock:
            for c in self.available:
                c.close()
            self.available.clear()

### Approach 2 — Optimal (semaphore for capacity, tiny lock for the stack, `open()` outside both)

**Idea:** split the three jobs onto the three right tools.

1. **`semaphore.acquire()`** — capacity control. Blocks when `max_size` connections are already out. No lock is held while blocked, so waiting threads cost nothing.
2. **A lock held for exactly one `pop()`** — microseconds. Two threads can never take the same idle connection.
3. **`conn.open()` with no lock held** — so N threads open N connections **concurrently**, in the time of one.

**Why the `try/except` around creation is not optional.** If `Connection()` or `open()` raises, the thread already holds a permit. Without releasing it, that permit is lost forever — after `max_size` failures the pool deadlocks permanently, and it looks like a hang, not a bug. The `except: release; raise` is the fix, and it is the follow-up the interviewer usually asks about.

**Why a stack (LIFO) and not a queue (FIFO)?** The most recently used connection is the most likely to still be warm — alive in the server's memory, TCP window opened up, not yet idle-timed-out. LIFO also lets genuinely surplus connections sit untouched at the bottom, which is exactly what an idle-eviction reaper wants to find.

**Time complexity:** O(1) per call, and the critical section is O(1) *stack operations* — no I/O inside it.

**Space complexity:** O(n) — at most `max_size` connections exist.

In [ ]:
class ConnectionPool:
    """Semaphore bounds checkouts; a tiny lock guards the stack; open() runs lock-free."""

    def __init__(self, max_size: int) -> None:
        self.max_size = max_size
        self.available: List[Connection] = []
        self.lock = threading.Lock()                 # guards `available` ONLY
        self.semaphore = threading.Semaphore(max_size)  # counts free capacity
        self.created = 0

    def get_connection(self) -> Connection:
        self.semaphore.acquire()                     # block if max_size are already out
        conn = None
        with self.lock:                              # critical section: ONE pop
            if self.available:
                conn = self.available.pop()          # LIFO: reuse the warmest connection
        if conn is None:
            try:
                conn = Connection()
                conn.open()                          # EXPENSIVE - deliberately outside every lock
                with self.lock:
                    self.created += 1
            except BaseException:
                self.semaphore.release()             # never leak a permit on failure
                raise
        return conn

    def release_connection(self, conn: Optional[Connection]) -> None:
        if conn is None:
            return
        with self.lock:
            self.available.append(conn)              # critical section: ONE push
        self.semaphore.release()                     # hand the permit back AFTER it is reusable

    def close_all(self) -> None:
        with self.lock:
            for c in self.available:
                c.close()
            self.available.clear()
            self.created = 0

### Follow-up — a production-shaped pool (timeout, validation, idle eviction)

**Idea:** three additions that turn the exercise into something like HikariCP.

- **`timeout`** — `semaphore.acquire(timeout=...)` returns `False` instead of blocking forever. A caller that waits indefinitely for a connection turns a slow database into a hung application, so real pools always expose this. Returning `False` (rather than raising deep inside) keeps the failure explicit at the call site.
- **Validation on checkout** — an idle connection may have been killed by the server, a firewall, or an idle timeout while it sat in the stack. Check it before handing it out; if it is dead, **discard and try the next one**, and fall through to creating a fresh connection if the stack empties. Note the permit is *not* released while discarding — the caller still gets a connection, so the checkout is still valid.
- **Idle eviction** — a connection unused for longer than `max_idle` is wasting a server-side resource. Storing `(connection, timestamp)` pairs lets a reaper close the stale ones. Because the stack is LIFO, the genuinely idle ones sink to the bottom, so the reaper naturally finds them there.

**Time complexity:** O(1) amortized per checkout (validation may discard a few dead connections); O(k) for one eviction sweep over k idle connections.

**Space complexity:** O(n).

In [ ]:
class ProductionConnectionPool:
    """Adds acquire timeouts, checkout validation and idle eviction."""

    def __init__(self, max_size: int, max_idle: float = 30.0) -> None:
        self.max_size = max_size
        self.max_idle = max_idle
        self.available: List[tuple] = []                 # (connection, returned_at)
        self.lock = threading.Lock()
        self.semaphore = threading.Semaphore(max_size)
        self.created = 0
        self.evicted = 0
        self.discarded = 0

    @staticmethod
    def _is_valid(conn: Connection) -> bool:
        return conn.opened and not conn.closed          # a real pool would ping the server

    def get_connection(self, timeout: Optional[float] = None) -> Optional[Connection]:
        if not self.semaphore.acquire(timeout=timeout):
            return None                                 # explicit failure beats an infinite wait
        try:
            while True:
                conn = None
                with self.lock:
                    if self.available:
                        conn, _ = self.available.pop()
                if conn is None:
                    break                               # stack empty -> create a fresh one
                if self._is_valid(conn):
                    return conn
                conn.close()                            # dead on arrival: drop it and retry
                self.discarded += 1
            conn = Connection()
            conn.open()
            with self.lock:
                self.created += 1
            return conn
        except BaseException:
            self.semaphore.release()
            raise

    def release_connection(self, conn: Optional[Connection]) -> None:
        if conn is None:
            return
        with self.lock:
            self.available.append((conn, time.monotonic()))
        self.semaphore.release()

    def evict_idle(self, now: Optional[float] = None) -> int:
        """Close connections idle for longer than max_idle. Returns how many were closed."""
        now = time.monotonic() if now is None else now
        with self.lock:
            keep, drop = [], []
            for conn, ts in self.available:
                (drop if now - ts > self.max_idle else keep).append(conn)
            self.available = [(c, t) for c, t in self.available if c in keep]
        for c in drop:                                  # close OUTSIDE the lock - it may be slow
            c.close()
        self.evicted += len(drop)
        return len(drop)

    def close_all(self) -> None:
        with self.lock:
            for c, _ in self.available:
                c.close()
            self.available.clear()

## Verification

Correctness under concurrency cannot be argued from reading the code — it has to be *run*. These checks hammer both pools with real threads and assert the invariants that matter: capacity is never exceeded, connections are opened exactly once, no connection is ever handed to two threads at the same time, and a failing `open()` does not leak a permit.

In [ ]:
import random
from concurrent.futures import ThreadPoolExecutor

# --- Basic reuse: a released connection is handed back out, not re-opened ---
reset_connection_counter()
pool = ConnectionPool(max_size=2)
c1 = pool.get_connection()
assert c1.opened and not c1.closed
pool.release_connection(c1)
c2 = pool.get_connection()
assert c2 is c1, "an idle connection must be reused, not recreated"
assert pool.created == 1
pool.release_connection(c2)
pool.close_all()
assert c1.closed

# --- Capacity is never exceeded, and no connection is ever double-issued ---
reset_connection_counter()
POOL_SIZE, THREADS, ROUNDS = 4, 16, 8
pool = ConnectionPool(max_size=POOL_SIZE)

live = set()
live_lock = threading.Lock()
max_live = [0]
errors = []


def worker(_):
    for _ in range(ROUNDS):
        conn = pool.get_connection()
        try:
            with live_lock:
                if conn in live:
                    errors.append("same connection issued to two threads at once")
                live.add(conn)
                max_live[0] = max(max_live[0], len(live))
            conn.write("x")
            assert conn.read().startswith("data-from-")
            time.sleep(random.uniform(0, 0.002))
            with live_lock:
                live.discard(conn)
        finally:
            pool.release_connection(conn)


with ThreadPoolExecutor(max_workers=THREADS) as ex:
    list(ex.map(worker, range(THREADS)))

assert not errors, errors
assert max_live[0] <= POOL_SIZE, f"{max_live[0]} checked out at once, max is {POOL_SIZE}"
assert pool.created <= POOL_SIZE, f"created {pool.created}, max is {POOL_SIZE}"
assert Connection._counter <= POOL_SIZE
pool.close_all()

# --- Concurrency really is concurrent: N opens take ~1 open, not N ---
reset_connection_counter()
pool = ConnectionPool(max_size=8)
start = time.monotonic()
with ThreadPoolExecutor(max_workers=8) as ex:
    conns = list(ex.map(lambda _: pool.get_connection(), range(8)))
parallel_elapsed = time.monotonic() - start
for c in conns:
    pool.release_connection(c)
assert parallel_elapsed < 8 * OPEN_COST * 0.6, (
    f"8 opens took {parallel_elapsed:.3f}s; they should overlap, not serialise"
)
pool.close_all()

# --- A failing open() must NOT leak a semaphore permit ---
boom = ConnectionPool(max_size=1)
original_open = Connection.open
calls = {"n": 0}


def flaky_open(self):
    calls["n"] += 1
    if calls["n"] == 1:
        raise IOError("handshake failed")
    original_open(self)


Connection.open = flaky_open
try:
    try:
        boom.get_connection()
    except IOError:
        pass
    else:
        raise AssertionError("the failure should propagate to the caller")
    # If the permit leaked, this call blocks forever - the timeout turns that into a failure.
    done = threading.Event()
    result = {}

    def try_again():
        result["conn"] = boom.get_connection()
        done.set()

    threading.Thread(target=try_again, daemon=True).start()
    assert done.wait(timeout=3.0), "permit leaked on a failed open(): the pool deadlocked"
    assert result["conn"].opened
finally:
    Connection.open = original_open

# --- Production pool: timeout, validation, eviction ---
reset_connection_counter()
prod = ProductionConnectionPool(max_size=1, max_idle=0.05)
held = prod.get_connection()
assert held is not None
assert prod.get_connection(timeout=0.05) is None, "a full pool must time out, not hang"
prod.release_connection(held)
assert prod.get_connection(timeout=0.05) is held
prod.release_connection(held)

# A connection killed while idle is discarded on checkout, not handed out
held.close()
fresh = prod.get_connection(timeout=1.0)
assert fresh is not held, "a dead connection must not be handed out"
assert prod.discarded == 1
prod.release_connection(fresh)

# Idle eviction closes what has sat unused past max_idle
time.sleep(0.08)
assert prod.evict_idle() == 1
assert fresh.closed
assert prod.available == []
prod.close_all()

# --- The naive pool gives the same ANSWERS - it is only slower ---
reset_connection_counter()
naive = NaiveConnectionPool(max_size=2)
n1 = naive.get_connection()
naive.release_connection(n1)
assert naive.get_connection() is n1
naive.close_all()

print("All assertions passed.")

## Discussion — remaining follow-up directions

- **`open()` throwing.** Covered in the code and asserted above, because it is the highest-value follow-up: the permit is acquired *before* the risky work, so the failure path must hand it back. Forgetting this produces a pool that works fine in testing and permanently deadlocks in production after `max_size` transient network errors — a hang with no stack trace, which is far worse than a crash.
- **Timeouts.** `semaphore.acquire(timeout=t)`. The design question is what to do on failure: return `None` (as here), raise a `PoolExhaustedError`, or fall back to an unpooled connection. Returning a value forces the caller to decide, which is usually right — a web request would rather fail fast with a 503 than pile up threads.
- **Idle eviction.** Implemented above as an explicit `evict_idle()` so it is testable; in production a daemon thread calls it on a timer. Two details matter: close connections **outside** the lock (closing can block on the network), and keep a `min_idle` floor so the pool does not evict everything during a lull and then pay full open cost when traffic returns.
- **Staleness in general.** Validation on checkout costs a round trip. Cheaper heuristics: trust connections younger than a `max_lifetime`, validate only if idle longer than some threshold, or validate *on release* so the latency lands off the request's critical path. All three are real HikariCP settings.
- **Dynamic resizing.** `threading.Semaphore` has no "shrink" operation. Growing is easy — call `release()` extra times. Shrinking is not, because permits may currently be held; you need a target-size variable plus a `Condition`, where `release_connection` checks "are we over target?" and closes the connection instead of returning it to the stack. Say that shrinking is the hard direction and why.
- **Double-checked locking.** If asked: the classic broken idiom checks a field, takes a lock, checks again, then constructs. Without `volatile` (Java) the constructing thread's writes can be reordered so another thread sees a non-null but half-initialised object. The semaphore design sidesteps it entirely, because capacity is decided by the semaphore *before* any object exists — there is no "check, then maybe construct" to get wrong.

## Empirical complexity check

This is the measurement that makes the whole answer land. A fixed pool of 4 connections, and a growing number of threads each doing one checkout → work → release cycle. The only difference between the two pools is **where the lock sits**.

| Growth as thread count doubles | What it means |
|---|---|
| ~2x | serialised — threads are queueing behind each other's `open()` |
| ~1x up to the pool size, then flat | genuinely concurrent — opens overlap, and after warm-up nobody opens at all |

The naive pool pays `20 ms` per open **in sequence**; the optimal pool overlaps the first four and then reuses them forever.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

from concurrent.futures import ThreadPoolExecutor

POOL_SIZE = 4


def make_workload(num_threads):
    return (num_threads,)


def _drive(pool, num_threads):
    def task(_):
        conn = pool.get_connection()
        try:
            conn.write("payload")
            conn.read()
        finally:
            pool.release_connection(conn)

    with ThreadPoolExecutor(max_workers=num_threads) as ex:
        list(ex.map(task, range(num_threads)))


def run_naive(num_threads):
    reset_connection_counter()
    _drive(NaiveConnectionPool(POOL_SIZE), num_threads)


def run_optimal(num_threads):
    reset_connection_counter()
    _drive(ConnectionPool(POOL_SIZE), num_threads)


benchmark(
    {"Approach 1 - lock held across open()": run_naive,
     "Approach 2 - semaphore + tiny lock": run_optimal},
    make_workload,
    sizes=[4, 8, 16, 32],
    repeats=1,
)

## Patterns learned

- **Never hold a lock across slow work.** I/O, network calls, `sleep`, or anything that can block belongs outside the critical section. This one rule explains most of the difference between a pool that scales and one that does not.
- **A lock answers "who may touch this?"; a semaphore answers "how many may proceed?"** Reaching for a mutex when the real constraint is a *count* leads to hand-rolled counters, `wait`/`notify`, and the bugs that come with them.
- **Acquire the resource, then guard the failure path.** Any `acquire()` before risky work needs a `try/except` that releases on the way out. A leaked permit is a silent, permanent deadlock — the worst failure mode there is, because there is nothing to see in the logs.
- **Pool anything whose creation dwarfs its use.** Connections, threads, buffers, parsed regexes, ML model handles. The pattern is always the same: bound the count, reuse the instances, validate before handing out.
- **LIFO for reuse, FIFO for fairness.** Reuse the *warmest* item (stack) when items go stale with age; use a queue when you care that everyone gets a turn. Knowing which axis you are optimising is the point.
- **Concurrency claims must be tested, not argued.** The assertions above — max simultaneous checkouts, no double-issue, no permit leak, opens actually overlapping — are the kind of proof that reading the code cannot give you.